# 1.8 Regularization theory

Add a penalty on complexity to the training objective: the smooth, continuous cousin of Structural Risk Minimization.

Regularization replaces SRM's discrete class ladder with a continuous strength $\lambda$. The ridge path below shows how the norm shrinks, how risk moves, and why scaling and intercept handling matter.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(16)


def theory_ladder(topic):
    """Return compact D1--D5 inputs for F16 theory notebooks."""
    if topic == "srm":
        return [
            {"name": "D1 lesson four-rung ladder", "emp": np.array([0.30, 0.15, 0.07, 0.05]), "H": np.array([2, 16, 256, 65536]), "m": 200, "delta": 0.05},
            {"name": "D2 six nested polynomial rungs", "emp": np.array([0.34, 0.21, 0.13, 0.09, 0.075, 0.07]), "H": np.array([2, 8, 32, 128, 512, 2048]), "m": 200, "delta": 0.05},
            {"name": "D3 larger sample shifts upward", "emp": np.array([0.30, 0.15, 0.07, 0.04, 0.035]), "H": np.array([2, 16, 256, 65536, 1048576]), "m": 2000, "delta": 0.05},
            {"name": "D4 noisy empirical errors", "emp": np.array([0.31, 0.19, 0.12, 0.115, 0.13]), "H": np.array([2, 16, 256, 4096, 65536]), "m": 300, "delta": 0.05},
            {"name": "D5 bad ordering stress case", "emp": np.array([0.28, 0.10, 0.03, 0.02, 0.00]), "H": np.array([4, 4096, 64, 1048576, 256]), "m": 120, "delta": 0.05},
        ]
    if topic == "regularization":
        x = np.linspace(-1.0, 1.0, 25)
        base = 1.0 + 2.0 * x - 1.5 * x ** 2
        return [
            {"name": "D1 lesson 3-row regression", "X": np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]]), "y": np.array([1.0, 2.0, 2.0]), "lambdas": np.array([0.0, 1.0, 10.0, 100.0])},
            {"name": "D2 dense lambda path", "X": np.column_stack([np.ones_like(x), x]), "y": 1.0 + 2.0 * x + 0.15 * np.sin(7.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D3 higher polynomial capacity", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3, x ** 4, x ** 5]), "y": base + 0.10 * np.sin(9.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D4 noisy labels", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3]), "y": base + rng.normal(0.0, 0.18, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D5 unscaled-feature trap", "X": np.column_stack([np.ones_like(x), x, 100.0 * x ** 2]), "y": base + rng.normal(0.0, 0.10, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
        ]
    if topic == "stability":
        sample = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
        long_sample = np.linspace(0.0, 1.0, 50)
        return [
            {"name": "D1 lesson 5-number mean", "sample": sample, "kind": "mean"},
            {"name": "D2 m=50 bounded-range mean", "sample": long_sample, "kind": "mean"},
            {"name": "D3 ridge lambda ladder", "sample": np.column_stack([np.ones(20), np.linspace(-1.0, 1.0, 20)]), "target": 1.0 + np.linspace(-1.0, 1.0, 20), "lambdas": np.array([0.1, 0.3, 1.0, 3.0]), "kind": "ridge"},
            {"name": "D4 compare mean/ridge to 1-NN-style rule", "sample": np.linspace(0.0, 1.0, 20), "kind": "nn_compare"},
            {"name": "D5 outlier removal stress case", "sample": np.array([0.0, 0.1, 0.2, 0.3, 9.0]), "kind": "outlier"},
        ]
    if topic == "nfl":
        return [
            {"name": "D1 one unseen point", "n": 1, "prediction": np.array([0])},
            {"name": "D2 lesson three unseen points", "n": 3, "prediction": np.array([0, 0, 0])},
            {"name": "D3 four unseen points", "n": 4, "prediction": np.array([1, 0, 1, 0])},
            {"name": "D4 biased smooth subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "smooth"},
            {"name": "D5 adversarial mirror subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "mirror"},
        ]
    if topic == "uniform":
        return [
            {"name": "D1 one fixed hypothesis", "H": 1, "m": 500, "epsilon": 0.1},
            {"name": "D2 hundred-hypothesis lesson class", "H": 100, "m": 500, "epsilon": 0.1},
            {"name": "D3 low-m vacuous rungs", "H": 100, "ms": np.array([100, 150, 230, 300]), "epsilon": 0.1},
            {"name": "D4 solve m for delta", "H": 100, "delta": 0.05, "epsilon": 0.1},
            {"name": "D5 correlated non-iid violation", "H": 100, "m": 500, "epsilon": 0.1, "rho": 0.9},
        ]
    raise ValueError(topic)


def all_binary_targets(n):
    rows = list(itertools.product([0, 1], repeat=n))
    return np.array(rows, dtype=int)


def ridge_weights(X, y, lam, penalize_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    penalty = np.eye(X.shape[1])
    if not penalize_intercept:
        penalty[0, 0] = 0.0
    system = X.T @ X + lam * penalty
    rhs = X.T @ y
    weights = np.linalg.solve(system, rhs)
    return weights


def print_rows(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, item in enumerate(row):
            widths[i] = max(widths[i], len(str(item)))
    header = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    print(header)
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(item).ljust(widths[i]) for i, item in enumerate(row)))


## The concept, built once (D1)

Ridge regression solves
$$\hat w=(X^\top X+\lambda I)^{-1}X^\top y$$
for the objective $\|Xw-y\|^2+\lambda\|w\|^2$. The lesson uses rows $[1,1],[1,2],[1,3]$ and $y=[1,2,2]$.

In [ ]:

def ridge_weights(X, y, lam, penalize_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    penalty = np.eye(X.shape[1])
    if not penalize_intercept:
        penalty[0, 0] = 0.0
    system = X.T @ X + lam * penalty
    rhs = X.T @ y
    weights = np.linalg.solve(system, rhs)
    return weights


def ridge_metrics(X, y, lam, penalize_intercept=True):
    weights = ridge_weights(X, y, lam, penalize_intercept=penalize_intercept)
    residual = X @ weights - y
    mse = float(np.mean(residual ** 2))
    norm_sq = float(np.sum(weights ** 2))
    objective = float(np.sum(residual ** 2) + lam * norm_sq)
    return weights, mse, norm_sq, objective


The exact lesson path is $\lambda=0,1,10,100$: weights near $[0.667,0.500]$, $[0.375,0.583]$, $[0.196,0.409]$, and squared norms $0.694,0.481,0.206,0.011$.

In [ ]:

X_lesson = np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]])
y_lesson = np.array([1.0, 2.0, 2.0])
lesson_lambdas = np.array([0.0, 1.0, 10.0, 100.0])
lesson_rows = []
lesson_norms = []
for lam in lesson_lambdas:
    weights, mse, norm_sq, objective = ridge_metrics(X_lesson, y_lesson, lam)
    lesson_norms.append(norm_sq)
    lesson_rows.append((lam, np.round(weights, 3).tolist(), f"{norm_sq:.3f}", f"{mse:.3f}"))
print_rows(lesson_rows, ["lambda", "weights", "norm_sq", "mse"])
assert np.allclose(ridge_weights(X_lesson, y_lesson, 0.0), np.array([2.0 / 3.0, 0.5]))
assert np.allclose(np.round(ridge_weights(X_lesson, y_lesson, 1.0), 3), np.array([0.375, 0.583]))
assert np.allclose(np.round(ridge_weights(X_lesson, y_lesson, 10.0), 3), np.array([0.196, 0.409]))
assert np.allclose(np.round(lesson_norms, 3), np.array([0.694, 0.481, 0.206, 0.011]))


## The dataset ladder

D1 is the exact lesson regression. D2 traces a dense $\lambda$ path, D3 adds polynomial capacity, D4 adds noise, and D5 creates the unscaled-feature trap.

In [ ]:

reg_ladder = theory_ladder("regularization")
rows = []
for dataset in reg_ladder:
    rows.append((dataset["name"], dataset["X"].shape, len(dataset["lambdas"]), np.round(dataset["y"][:3], 3).tolist()))
print_rows(rows, ["dataset", "X shape", "lambda count", "y sample"])


In [ ]:

reg_results = []
for dataset in reg_ladder:
    n = dataset["X"].shape[0]
    split = max(2, int(0.7 * n))
    X_train = dataset["X"][:split]
    y_train = dataset["y"][:split]
    X_valid = dataset["X"][split:]
    y_valid = dataset["y"][split:]
    rows = []
    path = []
    for lam in dataset["lambdas"]:
        weights, train_mse, norm_sq, objective = ridge_metrics(X_train, y_train, lam)
        valid_mse = float(np.mean((X_valid @ weights - y_valid) ** 2)) if len(y_valid) else train_mse
        gap = valid_mse - train_mse
        path.append((lam, train_mse, valid_mse, gap, norm_sq, weights))
    reg_results.append({"dataset": dataset, "path": path})
    for lam, train_mse, valid_mse, gap, norm_sq, weights in path[::max(1, len(path) // 4)]:
        rows.append((f"{lam:.3g}", f"{train_mse:.3f}", f"{valid_mse:.3f}", f"{gap:.3f}", f"{norm_sq:.3f}"))
    print(f"\n{dataset['name']}")
    print_rows(rows, ["lambda", "train", "valid", "gap", "norm_sq"])


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for j, result in enumerate(reg_results):
    dataset = result["dataset"]
    path = result["path"]
    x_axis = dataset["X"][:, 1]
    order = np.argsort(x_axis)
    for idx in np.linspace(0, len(path) - 1, 4, dtype=int):
        lam, train_mse, valid_mse, gap, norm_sq, weights = path[idx]
        axes[0, j].plot(x_axis[order], (dataset["X"] @ weights)[order], label=f"{lam:.2g}")
    axes[0, j].scatter(x_axis, dataset["y"], s=14, color="black")
    axes[0, j].set_title(dataset["name"].split()[0])
    lambdas = np.array([item[0] for item in path])
    norms = np.array([item[4] for item in path])
    gaps = np.array([item[3] for item in path])
    axes[1, j].semilogx(lambdas + 1e-12, norms, marker="o", label="norm")
    axes[1, j].semilogx(lambdas + 1e-12, np.maximum(gaps, 0.0), marker="o", label="gap")
    axes[1, j].set_xlabel("lambda")
    axes[1, j].set_ylabel("norm / risk gap")
axes[0, 0].legend()
axes[1, 0].legend()
fig.suptitle("Regularization capacity panels and norm/risk-vs-lambda curves")
plt.tight_layout()


## Pitfall on D5: skipping feature scaling and penalizing the intercept

The same $\lambda$ can punish coefficients unevenly when feature scales differ, and shrinking the intercept drags predictions toward the origin. Standardize non-intercept features and exclude the intercept from the penalty.

In [ ]:

d5 = reg_ladder[-1]
lam = 1.0
bad_weights, bad_train, bad_norm, bad_objective = ridge_metrics(d5["X"], d5["y"], lam, penalize_intercept=True)
X_fixed = d5["X"].copy()
feature_mean = X_fixed[:, 1:].mean(axis=0)
feature_std = X_fixed[:, 1:].std(axis=0)
X_fixed[:, 1:] = (X_fixed[:, 1:] - feature_mean) / feature_std
fixed_weights, fixed_train, fixed_norm, fixed_objective = ridge_metrics(X_fixed, d5["y"], lam, penalize_intercept=False)
print(f"Bad unscaled/intercept-penalized MSE: {bad_train:.3f}")
print(f"Fixed scaled/no-intercept-penalty MSE: {fixed_train:.3f}")
print(f"Bad weights: {np.round(bad_weights, 3)}")
print(f"Fixed weights: {np.round(fixed_weights, 3)}")


## Evaluate it + practice

- Metric: training loss plus norm, with validation risk gap along the lambda path.
- No-skill baseline: lambda equals zero plain least squares or a constant predictor at huge lambda.
- Cheap sanity check: squared norm should decrease as lambda increases.
- Ablation: remove the penalty and watch norm and validation gap rise on richer rungs.
- Failure signals: lambda is hand-set, intercept is shrunk, or features are not standardized.

### Practice prompts

- Try excluding the intercept in the lesson D1 calculation and compare norms.

- Add a lambda grid between 0.1 and 3 and find the best validation risk.

- Rescale one feature by 10 and predict which coefficient changes.
